# CS246 - Colab 3
## K-Means & PCA

### Setup

Let's set up Spark on your Colab environment.  Run the cell below!

In [ ]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq
!pip install pyspark findspark
!pip install -U -q PyDrive

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

import findspark
findspark.init()

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


Now we import some of the libraries usually needed by our workload.





In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import pyspark
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark import SparkContext, SparkConf

Let's initialize the Spark context.

In [ ]:
# create the session
conf = SparkConf().set("spark.ui.port", "4050")

# create the context
sc = pyspark.SparkContext(conf=conf)
spark = SparkSession.builder.getOrCreate()

### Data Preprocessing

In this Colab, rather than downloading a file from Google Drive, we will load a famous machine learning dataset, the [Breast Cancer Wisconsin dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html), using the ```scikit-learn``` datasets loader.

In [ ]:
from sklearn.datasets import load_breast_cancer
breast_cancer = load_breast_cancer()

For convenience, given that the dataset is small, we first

*   construct a Pandas dataframe
*   tune the schema
*   and convert it into a Spark dataframe.

In [ ]:
pd_df = pd.DataFrame(breast_cancer.data, columns=breast_cancer.feature_names)
df = spark.createDataFrame(pd_df)

def set_df_columns_nullable(spark, df, column_list, nullable=False):
    for struct_field in df.schema:
        if struct_field.name in column_list:
            struct_field.nullable = nullable
    df_mod = spark.createDataFrame(df.rdd, df.schema)
    return df_mod

df = set_df_columns_nullable(spark, df, df.columns)
df = df.withColumn('features', array(df.columns))
vectors = df.rdd.map(lambda row: Vectors.dense(row.features))

df.printSchema()

root
 |-- mean radius: double (nullable = false)
 |-- mean texture: double (nullable = false)
 |-- mean perimeter: double (nullable = false)
 |-- mean area: double (nullable = false)
 |-- mean smoothness: double (nullable = false)
 |-- mean compactness: double (nullable = false)
 |-- mean concavity: double (nullable = false)
 |-- mean concave points: double (nullable = false)
 |-- mean symmetry: double (nullable = false)
 |-- mean fractal dimension: double (nullable = false)
 |-- radius error: double (nullable = false)
 |-- texture error: double (nullable = false)
 |-- perimeter error: double (nullable = false)
 |-- area error: double (nullable = false)
 |-- smoothness error: double (nullable = false)
 |-- compactness error: double (nullable = false)
 |-- concavity error: double (nullable = false)
 |-- concave points error: double (nullable = false)
 |-- symmetry error: double (nullable = false)
 |-- fractal dimension error: double (nullable = false)
 |-- worst radius: double (nullable

With the next cell, we build the two data structures that we will be using throughout this Colab:


*   ```features```, a dataframe of Dense vectors, containing all the original features in the dataset;
*   ```labels```, a series of binary labels indicating if the corresponding set of features belongs to a subject with breast cancer, or not.



In [ ]:
from pyspark.ml.linalg import Vectors
features = spark.createDataFrame(vectors.map(Row), ["features"])
labels = pd.Series(breast_cancer.target)

### Your task

If you run successfully the Setup and Data Preprocessing stages, you are now ready to cluster the data with the [K-means](https://spark.apache.org/docs/latest/ml-clustering.html) algorithm included in MLlib (Spark's Machine Learning library).
Set the ```k``` parameter to **2**, fit the model, and the compute the [Silhouette score](https://en.wikipedia.org/wiki/Silhouette_(clustering)) (i.e., a measure of quality of the obtained clustering).  

**IMPORTANT:** use the MLlib implementation of the Silhouette score (via ```ClusteringEvaluator```).

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Initialize KMeans model with k=2
kmeans = KMeans().setK(2).setSeed(1)
model = kmeans.fit(features)

# Make predictions
predictions = model.transform(features)

# Evaluate clustering by computing Silhouette score
evaluator = ClusteringEvaluator()
silhouette = evaluator.evaluate(predictions)
print(f"Silhouette with squared euclidean distance = {silhouette}")

Silhouette with squared euclidean distance = 0.8342904262826145


Take the predictions produced by K-means, and compare them with the ```labels``` variable (i.e., the ground truth from our dataset).  

Compute how many data points in the dataset have been clustered correctly (i.e., positive cases in one cluster, negative cases in the other).

*HINT*: you can use ```np.count_nonzero(series_a == series_b)``` to quickly compute the element-wise comparison of two series.

**IMPORTANT**: K-means is a clustering algorithm, so it will not output a label for each data point, but just a cluster identifier!  As such, label ```0``` does not necessarily match the cluster identifier ```0```.


In [ ]:
import numpy as np
import builtins

# Convert Spark predictions column to a Pandas Series
pred_series = pd.Series([row['prediction'] for row in predictions.select('prediction').collect()])

# Since cluster labels are arbitrary, check both mappings:
# 1) Prediction 0 -> Label 0, Prediction 1 -> Label 1
match_direct = np.count_nonzero(pred_series == labels)

# 2) Prediction 0 -> Label 1, Prediction 1 -> Label 0 (inverted)
match_inverted = np.count_nonzero((1 - pred_series) == labels)

# Use Python's built-in max to avoid conflict with PySpark SQL max function
correct_count = builtins.max(match_direct, match_inverted)

print(f"Number of correctly clustered points: {correct_count} out of {len(labels)}")
print(f"Accuracy: {correct_count / len(labels):.2%}")

Number of correctly clustered points: 486 out of 569
Accuracy: 85.41%


Now perform dimensionality reduction on the ```features``` using the [PCA](https://spark.apache.org/docs/latest/ml-features.html#pca) statistical procedure, available as well in MLlib.

Set the ```k``` parameter to **2**, effectively reducing the dataset size of a **15X** factor.

In [ ]:
from pyspark.ml.feature import PCA

# Initialize PCA with k=2
pca = PCA(k=2, inputCol="features", outputCol="pcaFeatures")
pca_model = pca.fit(features)

# Transform the features
pcaFeatures = pca_model.transform(features).select("pcaFeatures")
pcaFeatures.show(5, truncate=False)

+----------------------------------------+
|pcaFeatures                             |
+----------------------------------------+
|[-2260.013886292542,-187.9603012226371] |
|[-2368.9937557820544,121.58742425815514]|
|[-2095.6652015478608,145.113985658701]  |
|[-692.6905100570509,38.57692259208171]  |
|[-2030.2124927427067,295.2979839927931] |
+----------------------------------------+
only showing top 5 rows


Now run K-means with the same parameters as above, but on the ```pcaFeatures``` produced by the PCA reduction you just executed.

Compute the Silhouette score, as well as the number of data points that have been clustered correctly.

In [ ]:
pcaKmeans = KMeans().setK(2).setSeed(1).setFeaturesCol("pcaFeatures")
pcaKmeansModel = pcaKmeans.fit(pcaFeatures)

# Make predictions
pkmPredictions = pcaKmeansModel.transform(pcaFeatures)

# Evaluate clustering by computing Silhouette score
new_evaluator = ClusteringEvaluator(featuresCol="pcaFeatures")
new_silhouette = new_evaluator.evaluate(pkmPredictions)
print(f"Silhouette with squared euclidean distance = {new_silhouette}")

Silhouette with squared euclidean distance = 0.8348610363444832


Comparing new model to labels

In [ ]:
import numpy as np
import builtins

# Convert Spark predictions column to a Pandas Series
pkm_pred_series = pd.Series([row['prediction'] for row in pkmPredictions.select('prediction').collect()])

# Since cluster labels are arbitrary, check both mappings:
# 1) Prediction 0 -> Label 0, Prediction 1 -> Label 1
pkm_match_direct = np.count_nonzero(pkm_pred_series == labels)

# 2) Prediction 0 -> Label 1, Prediction 1 -> Label 0 (inverted)
pkm_match_inverted = np.count_nonzero((1 - pkm_pred_series) == labels)

# Use Python's built-in max to avoid conflict with PySpark SQL max function
pkm_correct_count = builtins.max(pkm_match_direct, pkm_match_inverted)

print(f"Number of correctly clustered points: {pkm_correct_count} out of {len(labels)}")
print(f"Accuracy: {pkm_correct_count / len(labels):.2%}")

Number of correctly clustered points: 486 out of 569
Accuracy: 85.41%


Once you obtained the desired results, **head over to Gradescope and submit your solution for this Colab**!